# 04 — Instruments and graphs: design knowledge as assumptions

Sometimes you know more than “treatment precedes outcome.” An **encouragement
design** randomizes an *offer* of a program; people choose whether to enroll;
you care about the effect of *enrollment* on employment.

This notebook loads a synthetic encouragement trial, encodes the design with
`Assumptions.iv` and with an explicit `CausalDiagram`, contrasts that with
point identification under an observed confounder, and briefly treats
measurement error on the outcome.


## The encouragement design

Columns:

- `offer` — randomized offer of training (instrument)
- `enrolled` — whether the person took up training (treatment)
- `employed` — employment at follow-up (outcome)

**Assumptions of the classic IV diagram:** the offer affects employment only
through enrollment (exclusion), and unobserved confounding may still link
enrollment and employment.


In [1]:
from pathlib import Path

def tutorial_data(name: str) -> Path:
    """Resolve notebooks/data/<name> whether __file__ is set (CI) or not."""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = [
        here / "data" / name,
        here / "notebooks" / "data" / name,
        Path.cwd() / "notebooks" / "data" / name,
        Path.cwd() / "data" / name,
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"could not find notebooks/data/{name}; tried {candidates}")

import pandas as pd
from partialid import (
    Assumptions, CausalDiagram, Query, Variables, bound, from_dataframe,
    identify_query, is_identified,
)  # is_identified(g, treatments, outcomes); identify_query(V, P, Q, a)

df = pd.read_csv(tutorial_data("encouragement_trial.csv"))
print(f"N = {len(df)}")
print(df.head(8))
print()
print("offer rate:", round(df["offer"].mean(), 3))
print("enrollment rate:", round(df["enrolled"].mean(), 3))
print()
print("First stage — enrollment by offer:")
print(df.groupby("offer")["enrolled"].mean().rename("P(enrolled=1 | offer)"))
print()
print("Employment by offer and enrollment:")
print(df.groupby(["offer", "enrolled"])["employed"].mean().unstack("enrolled").round(3))


N = 3000
   offer  enrolled  employed
0      0         1         0
1      1         1         0
2      1         0         0
3      0         0         0
4      0         0         0
5      1         1         0
6      0         0         0
7      1         0         1

offer rate: 0.244
enrollment rate: 0.446

First stage — enrollment by offer:
offer
0    0.374780
1    0.665301
Name: P(enrolled=1 | offer), dtype: float64

Employment by offer and enrollment:
enrolled      0      1
offer                 
0         0.255  0.312
1         0.555  0.413


## Causal question: effect of enrollment, not offer


In [2]:
data = from_dataframe(df, columns=["offer", "enrolled", "employed"])
Q = Query.interventional(outcome="employed", do={"enrolled": 1}, value=1)
print(Q.describe())
print(
    "associational among enrolled:",
    round(df.loc[df["enrolled"] == 1, "employed"].mean(), 4),
)


P(employed=1 | do(enrolled=1))
associational among enrolled: 0.3485


## IV recipe

`Assumptions.iv(instrument, treatment, outcome)` builds the classic diagram
without asking you to write edges by hand.


In [3]:
a_iv = Assumptions.iv("offer", "enrolled", "employed")
r_iv = bound(data.variables, data.P, Q, a_iv)
print(r_iv.summary(Q))
print("assumption list:")
for line in a_iv.describe():
    print(" ", line)


query: P(employed=1 | do(enrolled=1))
interval: [0.27459, 0.60929]
width: 0.334699
method: lp
tight: True
success: True
message: lo:Optimization terminated successfully. (HiGHS Status 7: Optimal); hi:Optimization terminated successfully. (HiGHS Status 7: Optimal)
assumption list:
  DiagramAssumption: DiagramAssumption(diagram=CausalDiagram(nodes=('offer', 'enrolled', 'employed'), directed=(('offer', 'enrolled'), ('enrolled', 'employed')), bidirected=(('employed', 'enrolled'),)))


## Compliance monotonicity (no-defiers)

`monotone=True` adds **no-defiers**: nobody enrolls when not offered but refuses
when offered. That is a substantive claim. If it conflicts with the observed
law, the LP is **infeasible** (`success=False`) — that is informative, not a
crash.


In [4]:
a_mono = Assumptions.iv("offer", "enrolled", "employed", monotone=True)
r_mono = bound(data.variables, data.P, Q, a_mono)
print("success:", r_mono.success)
if r_mono.success:
    print(r_mono.summary(Q))
else:
    print("infeasible under no-defiers for this sample.")
    print("message:", r_mono.message)


success: False
infeasible under no-defiers for this sample.
message: lo:The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None); hi:The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)


## Same assumptions from an explicit diagram

Recipes are convenience. Equivalent structure via `CausalDiagram`:


In [5]:
g = CausalDiagram.from_edges(
    nodes=["offer", "enrolled", "employed"],
    directed=[("offer", "enrolled"), ("enrolled", "employed")],
    bidirected=[("enrolled", "employed")],
)
a_diag = Assumptions.from_diagram(g)
r_diag = bound(data.variables, data.P, Q, a_diag)
print(r_diag.summary(Q))
print(
    "recipe width matches diagram?",
    abs(r_iv.width - r_diag.width) < 1e-8,
)


query: P(employed=1 | do(enrolled=1))
interval: [0.27459, 0.60929]
width: 0.334699
method: lp
tight: True
success: True
message: lo:Optimization terminated successfully. (HiGHS Status 7: Optimal); hi:Optimization terminated successfully. (HiGHS Status 7: Optimal)
recipe width matches diagram? True


## Contrast: observed confounder → point identification

If the confounder is **observed**, classical identification can collapse the
interval to a point. Here a small Markovian back-door example (synthetic joint)
shows `method="id"` and width 0.

`is_identified` / `identify_query` are **experimental** APIs — useful for
checks, but prefer `bound` + reading `method` / `tight` in applications.


In [6]:
import numpy as np

V_bd = Variables.from_binary("risk", "enrolled", "employed")
g_bd = CausalDiagram.from_edges(
    ["risk", "enrolled", "employed"],
    directed=[("risk", "enrolled"), ("risk", "employed"), ("enrolled", "employed")],
    bidirected=[],
)
rng = np.random.default_rng(0)
keys = V_bd.cells()
w = np.exp(rng.normal(size=len(keys)))
w = w / w.sum()
P_bd = {k: float(v) for k, v in zip(keys, w)}
Q_bd = Query.interventional(outcome="employed", do={"enrolled": 1}, value=1)
a_bd = Assumptions.from_diagram(g_bd)

print(
    "is_identified (experimental):",
    is_identified(g_bd, ["enrolled"], ["employed"]),
)
id_res = identify_query(V_bd, P_bd, Q_bd, a_bd)
print("identify_query:", id_res.identified, "value=", id_res.value)

r_bd = bound(V_bd, P_bd, Q_bd, a_bd)
print(r_bd.summary(Q_bd))


is_identified (experimental): True
identify_query: True value= 0.3956808162722366
query: P(employed=1 | do(enrolled=1))
interval: [0.395681, 0.395681]
width: 0
method: id
tight: True
success: True
message: point-identified


## Ignoring the instrument widens the bounds

If we throw away `offer` and treat `(enrolled, employed)` as a plain
observational pair under Manski, we lose design information.


In [7]:
df_xy = df[["enrolled", "employed"]]
data_xy = from_dataframe(df_xy, columns=["enrolled", "employed"])
r_manski = bound(
    data_xy.variables, data_xy.P, Q,
    Assumptions.manski("enrolled", "employed"),
)
print(f"IV width:     {r_iv.width:.4f}")
print(f"Manski width: {r_manski.width:.4f}")
print("Design knowledge (the instrument) shrinks the identified set.")


IV width:     0.3347
Manski width: 0.5543
Design knowledge (the instrument) shrinks the identified set.


## IV + sensitivity budget

You can compose diagram structure with a quantitative confounding limit on
enrollment.


In [11]:
for lam in (2.0, 3.0):
    a_mix = a_iv & Assumptions.tan_msm("enrolled", lam)
    r_mix = bound(data.variables, data.P, Q, a_mix)
    if r_mix.success:
        print(f"Λ={lam:g}: [{r_mix.lower:.4f}, {r_mix.upper:.4f}]  width={r_mix.width:.4f}")
        print(f"  IV only width {r_iv.width:.4f} → IV+MSM width {r_mix.width:.4f}")
    else:
        print(f"Λ={lam:g}: infeasible (MSM too tight for this IV law)")
        print(f"  message: {r_mix.message}")


Λ=2: infeasible (MSM too tight for this IV law)
  message: lo:The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None); hi:The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
Λ=3: [0.3206, 0.4761]  width=0.1554
  IV only width 0.3347 → IV+MSM width 0.1554


## Measurement error (short case)

Sometimes the **true** outcome is latent and you observe a misclassified proxy.
Declare latent endogenous variables with `observed=...`: they still have
response types and may appear in queries, but they do **not** index `P`.


In [9]:
# Latent true employment; observe employed_obs. enrolled → employed → employed_obs,
# with confounding between true and measured employment.
V_me = Variables.from_binary(
    "enrolled", "employed", "employed_obs",
    observed=["enrolled", "employed_obs"],
)
g_me = CausalDiagram.from_edges(
    ["enrolled", "employed", "employed_obs"],
    directed=[("enrolled", "employed"), ("employed", "employed_obs")],
    bidirected=[("employed", "employed_obs")],
)
# Observational law on (enrolled, employed_obs) only — reuse Study A margins as a stand-in
P_me = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q_me = Query.interventional(outcome="employed", do={"enrolled": 1}, value=1)
a_me = Assumptions.from_diagram(g_me) & Assumptions.mtr("employed_obs", "employed")

r_me = bound(V_me, P_me, Q_me, a_me)
print(r_me.summary(Q_me))
print(f"tight={r_me.tight}  method={r_me.method}")

r_me0 = bound(V_me, P_me, Q_me, Assumptions.from_diagram(g_me))
print(f"diagram only width={r_me0.width:.4f}; + MTR on measurement channel width={r_me.width:.4f}")


query: P(employed=1 | do(enrolled=1))
interval: [0.3, 1]
width: 0.7
method: lp
tight: True
success: True
message: lo:Optimization terminated successfully. (HiGHS Status 7: Optimal); hi:Optimization terminated successfully. (HiGHS Status 7: Optimal)
tight=True  method=lp
diagram only width=1.0000; + MTR on measurement channel width=0.7000


## Always read `tight`

`tight=True` means the interval is sharp for the stated assumptions.
`tight=False` means a valid **outer** approximation (common for some
multi-component graphs). Warnings on the result explain when that happens.
Deeper routing (`classify_program`, `prefer_poly`) is covered in notebook 07.


In [10]:
print("IV tight:", r_iv.tight, "warnings:", r_iv.warnings)
print("back-door tight:", r_bd.tight, "method:", r_bd.method)


IV tight: True warnings: []
back-door tight: True method: id


## You can now…

- Encode an encouragement / IV design with `Assumptions.iv` or `CausalDiagram`.
- Handle infeasible monotonicity cleanly via `success` / `message`.
- See when an observed confounder point-identifies vs when IV only partially identifies.
- Compose IV structure with sensitivity budgets.
- Declare latent outcomes for simple measurement-error graphs.

**Next:** [05_queries_and_results.ipynb](05_queries_and_results.ipynb) — more
estimands, data ingress, and reporting. Optional deep dive:
[07_advanced_identification.ipynb](07_advanced_identification.ipynb).
